# Cria df positivo

# Cria df negativo

# Cria df neutro

# Cria df treinamento

# Raw Data para novo dataset a ser analisado

In [4]:
df1 = pd.read_csv('df_positivo.csv', encoding = 'utf-8', sep = ';')
df2 = pd.read_csv('df_negativo.csv', encoding = 'utf-8', sep = ';')
df3 = pd.read_csv('df_neutro.csv', encoding = 'utf-8', sep = ';')
df_total = pd.read_csv('df_total.csv', encoding = 'utf-8', sep = ';')
df_balance = pd.read_csv('df_balance.csv', encoding = 'utf-8', sep = ';')
df_novo = pd.read_csv('df_novo.csv', encoding = 'utf-8', sep = ';')

# Modelo Analise de Sentimentos Deep Learning

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import emoji
import spacy
import string
import tensorflow as tf
import unicodedata
import datetime
from sklearn.utils import shuffle
from joblib import dump, load
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import CSVLogger, TensorBoard, EarlyStopping
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split


def sentimento(x):
    """
    Função que transforma a coluna passada como argumento
    na variável target.
    """
    
    if x == 'POSITIVO':
        return 2
    elif x == 'NEUTRO':
        return 0
    else:
        return 1
    
    
def balance_classes(df):
    
    n3 = len(df[df['SENTIMENT'] == 'POSITIVO'])
    n2 = len(df[df['SENTIMENT'] == 'NEUTRO'])
    n1 = len(df[df['SENTIMENT'] == 'NEGATIVO'])
    aux_min = min(n1,n2,n3)
    #
    df1 = df.loc[df['SENTIMENT'] == 'POSITIVO', :]

    #
    df2 = df.loc[df['SENTIMENT'] == 'NEGATIVO', :]

    df3 = df.loc[df['SENTIMENT'] == 'NEUTRO', :]
   
    #
    df1 = df1.sample(aux_min)
    df2 = df2.sample(aux_min)
    df3 = df3.sample(aux_min)
    # contrói a tabela final
    df_final = df1.append(df2).reset_index().drop(columns=['index'])
    df_final = df_final.append(df3).reset_index().drop(columns=['index'])
    return df_final

def preprocess_data(data, columns,
                    label_column, null=True,
                    labels_ok=True, dict_labels=None,
                    labels_ready=True, labels_true=None):
    """ Função para tratamento de datasets para analise de sentimentos
    
    Args:
        data (pandas dataset) = Dataset 
        columns (list) = nome da coluna com texto para analise e da coluna com os labels
        null (Boolean) = define se vai limpar ou não os dados faltantes.
        label_columns (str)= nome da coluna com os label
        label_ok (Boolean) = define se a coluna de label está tratada ou não
        dict_label (dict) = define pelas chaves e valores as substituições vão ser feitas na coluna de labels.
        labels_ready (Boolean) = define se vamos usar os labels sem nenhuma alteração
        labels_true (dict) = dicionario com chaves sendo tuplas (com pelo menos dois elementos,
        mesmo que sejam elementos repetidos), com labels e valores sendo os respectivos 
        sentimentos (na ordem: positive, neutral e negative).
    """
    df = data[columns]
    
    if null:
        df = df.dropna().reset_index().drop(columns=['index'])
    '''
    if not labels_ok:
        for n, item in enumerate(df[label_column]):
            if type(item) == str:
                for key, value in dict_labels.items():
                    df[label_column][n] = re.sub(str(list(key)),value, df[label_column].iloc[n])
        df[label_column] = df[label_column].astype('float32')
    '''                
    if not labels_ready:
        def sentimento(x):
            if (x in [*labels_true.keys()][0]):
                return [*labels_true.values()][0]
            elif (x in [*labels_true.keys()][1]):
                return [*labels_true.values()][1]
            else:
                return [*labels_true.values()][2]
            
    df['sentimentos'] = df[label_column].apply(sentimento)    
    
    return df


def preprocess_text(text, remove_stop = True, 
                    stem_words = False, remove_mentions_hashtags = True):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # Corrige o bud que elimina parte das palavras com acentos
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()

    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  #pensar na  palavra 'OK'

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)


def tokenizacao(df):
    
    # cria a coluna com textos vetorizados
    rows, cols = df.shape

    df['token'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df


def treinar_vetorizacao(df):

    # coleciona as palavras usadas para o treinamento da função CountVectorizer
    lista_treino = []

    for item in df['token']:
        lista_treino1 = [n for n in item if n not in lista_treino]
        lista_treino.extend(lista_treino1)

    # treina o modelo de vetorização
    vectorize = CountVectorizer(lowercase=True, strip_accents='unicode')

    vectorize.fit(lista_treino)
    
    return vectorize, lista_treino


def vectorize2(lista):
    lista2 = [vectorize.vocabulary_[item] for item in lista]
    
    return lista2


def separa_df_test_train(df, size):

    #
    df = shuffle(df, random_state=42)

    # separando as variaveis
    X = df['vectors'].values

    #
    y = df['sentimentos'].values

    # usa a função pad_sequences do keras para deixar todos os textos do mesmo tamanho
    X = tf.keras.preprocessing.sequence.pad_sequences(X, maxlen=20)

    # separa os dados em treino e teste e fazendo o one-hot-encoding na variável y
    Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size = size, random_state=42)

    #
    ytrain = tf.keras.utils.to_categorical(ytrain)

    #
    ytest = tf.keras.utils.to_categorical(ytest)

    return X, y, Xtrain, Xtest, ytrain, ytest


def redes_neurais_modelo():

    #
    model = tf.keras.Sequential()

    #
    model.add(tf.keras.layers.Embedding(input_dim = 15*len(lista_treino) + 1, 
                                        output_dim = 128, 
                                        input_shape = (Xtrain.shape[1],), 
                                        activity_regularizer = regularizers.l2(1e-5)
                                       )
             )

    #
    model.add(tf.keras.layers.Dropout(0.25))

    #
    model.add(tf.keras.layers.LSTM(units = 512, activation='relu'))

    #
    model.add(tf.keras.layers.Dense(units = 256, activation='relu'))

    #
    model.add(tf.keras.layers.Dropout(0.25))

    #
    model.add(tf.keras.layers.Dense(units = 128, activation='relu'))

    #
    model.add(tf.keras.layers.Dropout(0.25))

    #
    model.add(tf.keras.layers.Dense(units = 64, activation='relu'))

    #
    model.add(tf.keras.layers.Dropout(0.25))

    #
    model.add(tf.keras.layers.Dense(units = 3, 
                                    activation='softmax', 
                                    activity_regularizer = regularizers.l2(1e-5)
                                   )
             )

    #
    model.compile(optimizer = 'Adam', 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy']
                 )

    #
    model.summary()

    return model


def cria_stopper_rede(epo):

    # criando um stopper para a rede
    stopper = EarlyStopping(monitor="val_accuracy",
                             patience=5, 
                            verbose=2, 
                            mode='max'
                           )

    callbacks = [stopper]

    #
    history = model.fit(x = Xtrain, 
                        y = ytrain, 
                        batch_size = 128,
                        validation_data = (Xtest, ytest),
                        epochs = epo,
                        callbacks = callbacks
                        )
    
    return callbacks, history

def feelings_prediction(model):
    
    # organiza os labels pelo valor da rating dado pelo cliente
    dict_feeling = {2: 'Positive', 1: 'Negative', 0: 'Neutral'}

    text = input("Digite seu texto aqui\n\n")
    
    text1 = preprocess_text(text)
    
    text1 = [vectorize2(text1)]
    
    text1 = tf.keras.preprocessing.sequence.pad_sequences(text1, maxlen=20)
    
    pred = model.predict(text1)
    
    predd = dict_feeling[np.argmax(pred)]
    
    print('\nSentimento analisado:', predd)
    
    
def feelings_prediction_v02(model,df):
    
    # organiza os labels pelo valor da rating dado pelo cliente
    dict_feeling = {2: 'Positive', 1: 'Negative', 0: 'Neutral'}
    
    lista = []
    
    for n, objeto in enumerate(df.token):
        diff = [item for item in objeto if item not in vectorize.vocabulary_.keys()]
        lista.extend(diff)
    
    for item in lista:
        vectorize.vocabulary_[item] = len(vectorize.vocabulary_) + 1
        
    df['vectors'] = df['token'].apply(vectorize2)
    
    text = df['vectors'].values
    
    text = tf.keras.preprocessing.sequence.pad_sequences(text, maxlen=20)
    
    lista_pred = model.predict(text)
    
    df['pred_sentiment'] = [dict_feeling[np.argmax(item)] for item in lista_pred]
    
    return df


##############################################################################################
######### Treina o modelo ##########################################################


# transformando e abrindo o arquivo em formato csv
#df = inputs[0]

portuguese_stopwords = nltk.corpus.stopwords.words('spanish')

#
df_final = preprocess_data(df_balance, 
                           columns=['COMMENT_TEXT', 'SENTIMENT'],
                           label_column='SENTIMENT', 
                           null = True,
                           labels_ok = False,
                           dict_labels = {(' '): '', (','): '.'},
                           labels_ready = False, 
                           #labels_true = {(5,5): 2, (4,4):0,(1,2,3):1}
                           labels_true = {('POSITIVO'): 2, ('NEUTRO'):0,('NEGATIVO'):1}
                          )

# contruindo a coluna nova com os sentimentos
df_final['sentimentos'] = df_final['SENTIMENT'].apply(sentimento)

#
df_final = tokenizacao(df_final)

#
vectorize, lista_treino = treinar_vetorizacao(df_final)

# criando a coluna com o texto vetorizado
df_final['vectors'] = df_final['token'].apply(vectorize2)


# faz o balance das classes
df_final = balance_classes(df_final)


# Criando datasets de teste e treino
X, y, Xtrain, Xtest, ytrain, ytest = separa_df_test_train(df_final, 0.25)

#
model = redes_neurais_modelo();

# criando um stopper para a rede
stopper = EarlyStopping(monitor="val_accuracy",
                         patience=15, 
                        verbose=1, 
                        mode='max'
                       )

callbacks = [stopper]

#
history = model.fit(x = Xtrain, 
                    y = ytrain, 
                    batch_size = 128,
                    validation_data = (Xtest, ytest),
                    epochs = 1,
                    callbacks = callbacks
                    )

##############################################################################################
######### Classifica o novo dataset ##########################################################

df2 = df_novo #inputs[1]

df2['SENTIMENT'] = ''


df2 = preprocess_data(df2, 
                           columns=['KEY','COMMENT_ID','COMMENT_TEXT', 'SENTIMENT'],
                           label_column='SENTIMENT', 
                           null = True,
                           labels_ok = False,
                           dict_labels = {(' '): '', (','): '.'},
                           labels_ready = False, 
                           labels_true = {('POSITIVO'): 2, ('NEUTRO'):0, ('NEGATIVO'):1}
                          )


# contruindo a coluna nova com os sentimentos
# df2['sentimentos'] = df2['SENTIMENT'].apply(sentimento)

#
df2 = tokenizacao(df2)

df3 = feelings_prediction_v02(model,df2)

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding (Embedding)        (None, 20, 128)           3663488   
_________________________________________________________________
dropout (Dropout)            (None, 20, 128)           0         
_________________________________________________________________
lstm (LSTM)                  (None, 512)               1312768   
_________________________________________________________________
dense (Dense)                (None, 256)               131328    
_________________________________________________________________
dropout_1 (Dropout)          (None, 256)               0         
_________________________________________________________________
dense_1 (Dense)              (None, 128)               32896     
_________________________________________________________________
dropout_2 (Dropout)          (None, 128)               0

# Cria Bag of Words

In [6]:
import pandas as pd
import numpy as np
import emoji
import string
import unicodedata
import nltk
import re

portuguese_stopwords = nltk.corpus.stopwords.words('spanish')

def preprocess_text(text, 
                    remove_stop = True, 
                    stem_words = False, 
                    remove_mentions_hashtags = True):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # Corrige o bud que elimina parte das palavras com acentos
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()
    
    

    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)
 

# carrega o dataset apenas com as colunas desejadas
df_bag = df_novo[['KEY','COMMENT_ID', 'COMMENT_TEXT']]

df_bag = df_bag.dropna().reset_index().drop(columns=['index'])

# cria a coluna com textos vetorizados
rows, cols = df_bag.shape

df_bag['WORD'] = [preprocess_text(df_bag["COMMENT_TEXT"][row]) for row in range(rows)]

lst_col = 'WORD'

df_bag = pd.DataFrame({col:np.repeat(df_bag[col].values, df_bag[lst_col].str.len())
    for col in df_bag.columns.difference([lst_col])}).assign(
    **{lst_col:np.concatenate(df_bag[lst_col].values)})[df_bag.columns.tolist()]

# Modelo Analise de Sentimentos Machine Learning

## ML - Classifica Sentimento para Label

In [7]:
import collections 
import pandas as pd
import numpy as np
import nltk

# df = inputs[0]

classificacao = df_balance["SENTIMENT"].replace(["NEGATIVO","NEUTRO","POSITIVO"], [-1,0,1])

df_balance["CLASSFICATION"] = classificacao

df_balance['COMMENT_TEXT'] = df_balance['COMMENT_TEXT'].str.replace('[^\w\s]','')

<ipython-input-7-6043a2a60da5>:12: FutureWarning: The default value of regex will change from True to False in a future version.
  df_balance['COMMENT_TEXT'] = df_balance['COMMENT_TEXT'].str.replace('[^\w\s]','')


## Tokenização Treino e Teste

In [8]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import re
import emoji
import unicodedata
import string

####################################
############## Funcoes #############

def preprocess_text(text, remove_stop = True, 
                    stem_words = False, remove_mentions_hashtags = True):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # Corrige o bud que elimina parte das palavras com acentos
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()

    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)

def tokenizacao(df):
    
    # cria a coluna com textos vetorizados
    rows, cols = df.shape

    df['TOKEN'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df

###################################################
############## Treina o modelo ####################
  
#carregando DataFrame
# df = inputs[0]

#carregando a lista de de Stopwords 
portuguese_stopwords = nltk.corpus.stopwords.words('spanish')

df = tokenizacao(df_balance)


## Vetorização Treino e Teste

In [9]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import  SGDClassifier
from sklearn.svm import SVC

####################################
############## Funcoes #############
  
def treinar_vetorizacao(df):

    # coleciona as palavras usadas para o treinamento da função CountVectorizer
    lista_treino = []

    for item in df['TOKEN']:
        lista_treino1 = [n for n in item if n not in lista_treino]
        lista_treino.extend(lista_treino1)

    # treina o modelo de vetorização
    vectorize = CountVectorizer(lowercase=True, strip_accents='unicode')

    vectorize.fit(lista_treino)
    
    return vectorize, lista_treino

def vectorize2(lista):
    lista2 = [vectorize.vocabulary_[item] for item in lista]
    
    return lista2

###################################################
############## Treina o modelo ####################
  
#carregando DataFrame
#df = inputs[0]
#
vectorize, lista_treino = treinar_vetorizacao(df)

# criando a coluna com o texto vetorizado
df['VECTORS'] = df['TOKEN'].apply(vectorize2)


## Tokenização Novas Entradas

In [10]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import re
import emoji
import unicodedata
import string

####################################
############## Funcoes #############

def preprocess_text(text, remove_stop = True, 
                    stem_words = False, remove_mentions_hashtags = True):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # Corrige o bud que elimina parte das palavras com acentos
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()

    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)

def tokenizacao(df):
    
    # cria a coluna com textos vetorizados
    rows, cols = df.shape

    df['TOKEN'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df

###################################################
############## Treina o modelo ####################
  
#carregando DataFrame
# df = inputs[0]

#carregando a lista de de Stopwords 
portuguese_stopwords = nltk.corpus.stopwords.words('spanish')

df = tokenizacao(df)


## Treino, Teste e Predição

In [11]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import  SGDClassifier
from sklearn.svm import SVC

####################################
############## Funcoes #############

# função para retornar apenas as palavras da frase, sem a classificação (sentimento)
def busca_Palavras(frases):
    todas_Palavras =[]
    for (palavras, sentimento) in frases:
        todas_Palavras.extend(palavras)
    return todas_Palavras
  
def busca_Palavras2(frases):
    todas_Palavras =[]
    for (palavras) in frases:
        todas_Palavras.extend(palavras)
    return todas_Palavras

# função para verificar a quantidade de vezes que a palavra é mencionada
def busca_frequencia(palavras):
    palavras = nltk.FreqDist(palavras)
    return palavras

# função para retornar somente as palavras únicas
def busca_palavras_unicas(frequencia):
    freq = frequencia.keys()
    return freq

# função para identificar quais palavras únicas estão no documento passado
def extrator_palavras(documento):
    doc = set(documento)
    caracteristicas = {}
    for palavras in palavras_unicas_treino:
        caracteristicas['%s' % palavras] = (palavras in doc)
    return caracteristicas

def extrator_palavras_teste(documento):
    doc = set(documento)
    caracteristicas = {}
    for palavras in palavras_unicas_teste:
        caracteristicas['%s' % palavras] = (palavras in doc)
    return caracteristicas

###################################################
############## Treina o modelo ####################
  
# proporção da base de treino
size_treino=0.3

#carregando DataFrame
# df = inputs[0]
df = df[["TOKEN","CLASSFICATION"]]

treino, teste = train_test_split(df, test_size=size_treino)

treino = [tuple(x) for x in treino.values]
palavras_treino = busca_Palavras(treino)

teste = [tuple(x) for x in teste.values]
palavras_teste = busca_Palavras(teste)

frequencia_treino = busca_frequencia(palavras_treino)
#df_treino = pd.DataFrame(frequencia_treino.most_common(20))
#frequencia_treino.plot(30, cumulative = False)

frequencia_teste = busca_frequencia(palavras_teste)

palavras_unicas_treino = busca_palavras_unicas(frequencia_treino)
palavras_unicas_teste = busca_palavras_unicas(frequencia_teste)

base_completa_treino = nltk.classify.apply_features(extrator_palavras, treino)
base_completa_teste = nltk.classify.apply_features(extrator_palavras_teste, teste)

classificador = nltk.NaiveBayesClassifier.train(base_completa_treino)
tst = nltk.classify.accuracy(classificador, base_completa_teste)

################

def classifica_txt(df1):
  teste = df1[["TOKEN"]]
  comStem = [p for p in teste.values]
  testeStemming.append(str(comStem[0]))
  test = []
  probabilidade = 0
  novo = extrator_palavras(testeStemming)
  distribuicao = classificador.prob_classify(novo)
  result = 0
    
  for classe in distribuicao.samples():
    test.append((str(classe) + " : " + str(distribuicao.prob(classe))))
    if distribuicao.prob(classe) >= probabilidade:
      result = str(classe)
      data = {'Result': [(classe)]}
      probabilidade = distribuicao.prob(classe)
  
  return result

def classifica_df(df):
  # cria a coluna com textos vetorizados
  rows, cols = df.shape
  df['LABEL_PREDICTION'] = [classifica_txt(df["TOKEN"][row]) for row in range(rows)]
  return df


#############

df1 = df #inputs[1]
teste = df1[["TOKEN"]]

#stemmer = nltk.stem.SnowballStemmer('portuguese')
#stemmer = nltk.stem.RSLPStemmer()
'''
testeStemming = []
comStem = [p for p in teste.values]
testeStemming.append(str(comStem[0]))
test = []
probabilidade = 0
novo = extrator_palavras(testeStemming)
distribuicao = classificador.prob_classify(novo)
for classe in distribuicao.samples():
  test.append((str(classe) + " : " + str(distribuicao.prob(classe))))
  if distribuicao.prob(classe) >= probabilidade:
	data = {'Result': [str(classe)]}
	probabilidade = distribuicao.prob(classe) 
'''
df1['SENTIMENT_PREDICTION'] = 'Nan'

for index, row in df1.iterrows():
  testr = row["TOKEN"]
  testr = str(' '.join(testr))
  teste = 'ruim pessimo'
  testeStemming = []
  comStem = ''
  for(palavras_treino) in testr.split(): 
  	comStem = [p for p in palavras_treino.split()]
  	testeStemming.append(str(comStem[0]))
  test = []
  probabilidade = 0
  novo = extrator_palavras(testeStemming)
  distribuicao = classificador.prob_classify(novo)
  result = ''
  for classe in distribuicao.samples():
    if distribuicao.prob(classe) >= probabilidade:
      probabilidade = distribuicao.prob(classe)
      result = str(classe).replace("-1", "Negative").replace("0", "Neutral").replace("1", "Positive")
      data = {'Result': [str(classe)]}
      df1.loc[index,'SENTIMENT_PREDICTION'] = result
      #test.append((str(classe) + " : " + str(distribuicao.prob(classe))))
    #if distribuicao.prob(classe) >= probabilidade:
	#  result = str(classe)
	#data = {'Result': [str(classe)]}
	#  probabilidade = distribuicao.prob(classe)
  #df1.loc[index,'LABEL_PREDICTION'] = result
	
# criando a coluna com o texto vetorizado
#df1['LABEL_PREDICTION'] = df1['TOKEN'].apply(classifica)
#df1 = classifica_df(df1)

#df1['LABEL_PREDICTION'] = df1['TOKEN'].apply(classifica_txt)
#df1['LABEL_PREDICTION'] = [classifica_txt(df1["TOKEN"][row]) for row in range(rows)]
  
  #data = {'Result': ['Probabilidade de ser negativa']}
	#test.append(str(class, distribuicao.prob(classe)))
	
'''	print('%s: %f' % (classe, distribuicao.prob(classe)))
    if classe == -1:
        data = {'Result': ['Probabilidade de ser negativa']}
	elif classe == 0:
    	data = {'Result': ['Probabilidade de ser Neutra']}
    else:
        data = {'Result': ['Probabilidade de ser Positiva']}
'''
#df = pd.DataFrame(data)

<ipython-input-11-b563f0bde717>:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['SENTIMENT_PREDICTION'] = 'Nan'
C:\ProgramData\Anaconda3\lib\site-packages\pandas\core\indexing.py:1720: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


"\tprint('%s: %f' % (classe, distribuicao.prob(classe)))\n    if classe == -1:\n        data = {'Result': ['Probabilidade de ser negativa']}\n\telif classe == 0:\n    \tdata = {'Result': ['Probabilidade de ser Neutra']}\n    else:\n        data = {'Result': ['Probabilidade de ser Positiva']}\n"